# ADS v2: Home MTA / cancellation / arrears / claims / document events

Builds 37 event families: `HM1`, `HM1.F1`, `HM2.O1`, `HM3`, `HM4`, `HM4a`, `HM4c`, `HM6`, `HM6.B1.1`, `HM6.B1.2`, `HM4.B1.F1`, `HM4.B1.1`, `HM4.B1.4`, `HM4.B1.5`, `HM4.B1.2`, `HM4.B1.3`, `HM4.B1.6`, `HM4.B1.O1`, `HC1a`, `HC1a.F1`, `HC2`, `HC3`, `HC4`, `HC5`, `HC6`, `HARR.1`, `HARR.2`, `HARR.O1`, `HARR.3`, `HARR.O2`, `HCL1a`, `HCL1b`, `HCL1c`, `HCL.O2`, `HD1`, `HD1.F1`, `HD3.F1`.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("16_Home_InLife", "started")


## Shared: hinl_mta

Home MTA postings (04 cell 34, stg.JJulyMTAs, without the July snapshot join): Relay accounts postings mapped to premium type group MTA with 04's posting filter, on Home policies, one row per policy and posting date with the gross premium and fees moved.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_hinl_mta                                                                    */
/*  Description : One row per Home policy and MTA posting date (04 cell 34 filter), with the gross prem */
/*              : ium and fees moved.                                                                   */
/*              :                                                                                       */
/*              : Reads: ods_v2.PolicyTransaction, ods_v2.Policy                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_hinl_mta as
select  t.PolicyCode,
        max(p.PolicyId)                                                 as PolicyId,
        max(t.ClientCode)                                               as ClientCode,
        t.PostingDate,
        cast(t.PostingDate as timestamp)                                as PostingDateTime,
        sum(coalesce(t.InsurerInitialNetPremium, 0) + coalesce(t.InsurerInitialCommission, 0)) as GrossPremium,
        sum(case when t.ClientTransType = '8' and t.DType = 5 then coalesce(t.ClientAmount, 0)
                 else coalesce(t.ClientFees, 0) end)                    as Fees
from    ${ads.ods}.PolicyTransaction t
join    ${ads.ods}.Policy p     on p.PolicyCode = t.PolicyCode
where   t.PremiumTypeGroup = 'MTA'
and     p.PolicyTypeGroup = 'Home'
and     cast(t.PostingDate as timestamp) >= timestamp '${ads.start_time}' and cast(t.PostingDate as timestamp) < timestamp '${ads.end_time}'
and     (   (t.InsurerTransType in ('X', 'Y', 'Z', '1', '3', '5', 'C', 'E') and t.TransSeq = 0)
         or (t.ClientTransType = '1' and t.NewFlag in (0, 4) and coalesce(trim(t.InsurerStatus), '') = ''
             and t.TransSeq = 0 and t.InsCoCode = 'ZZZZ')
         or (t.ClientTransType = '5' and t.ReturnSubType = '5' and t.TransSeq = 0 and t.InsCoCode = 'ZZZZ')
         or (t.ClientTransType = '8' and t.DType in (5, 6) and t.TransSeq = 0)
         or (t.InsCoCode = 'KCI' and t.AddonRef <> 0 and t.ShowOnReport = 'Y'))
group by t.PolicyCode, t.PostingDate

## Shared: hinl_chase

The Home MyChill chase snapshot (all policy types) with each row's own send date (D-E1.16; 04 stamped every chase at the sale date) and whether one of the six / seven documents is outstanding.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_hinl_chase                                                                  */
/*  Description : Home chase snapshot rows with their send date (DAY 1 / 7 / 14 / 20) and the six / sev */
/*              : en outstanding flags.                                                                 */
/*              :                                                                                       */
/*              : Reads: ods_v2.DocumentChaseHome                                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_hinl_chase as
select  PolicyCode,
        PolicyType,
        Campaign,
        cast(case Campaign when 'DAY 1'  then try_cast(Day1SendDate as date)
                           when 'DAY 7'  then try_cast(Day7SendDate as date)
                           when 'DAY 14' then try_cast(Day14SendDate as date)
                           when 'DAY 20' then try_cast(Day20SendDate as date) end as timestamp) as SendDate,
        case when (Gap_In_Cov_Ltr_Status = 'O' or Val_For_Spec_Item_Status = 'O' or PPS_Num_Status = 'O' or Identification_Status = 'O' or Finance_Form_Status = 'O' or Digital_Journey_Status = 'O') then 1 else 0 end                     as SixOutstanding,
        case when (Comp_DDM_Status = 'O' or Gap_In_Cov_Ltr_Status = 'O' or Val_For_Spec_Item_Status = 'O' or PPS_Num_Status = 'O' or Identification_Status = 'O' or Digital_Journey_Status = 'O' or Finance_Form_Status = 'O') then 1 else 0 end              as SevenOutstanding
from    ${ads.ods}.DocumentChaseHome
where   PolicyTypeGroup = 'Home'

## Shared: hinl_call

Inbound conversations on the Home change, claims and document-request queues (04 cells 6, 48, 62, 67), one row per conversation: start, the caller's phone key, which queues it touched, its wrap-up code (the agent's code; time-outs and the default code count as uncoded), wait and abandoned. The caller's Home policy is the one open at the call date for the SCV key holding the caller's number (D-E1.24, 10_Common; X4: replaces the pii phone lists); several open keep the lowest policy code.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_hinl_call                                                                   */
/*  Description : One row per inbound conversation on the Home change, INBOUND_Claims or INBOUND_Docume */
/*              : nts_Out queues.                                                                       */
/*              :                                                                                       */
/*              : Reads: ods_v2.CallSession                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_hinl_call as
select  ConversationId,
        min(ConversationStartTime)                                      as ConversationStartTime,
        max(AniPhoneKey)                                                as CustomerPhoneKey,
        max(case when QueueName in ('INBOUND_HomeChangeOther', 'INBOUND_HomeChangeBuildCost') then 1 else 0 end) as HomeChangeQueue,
        max(case when QueueName = 'INBOUND_Claims' then 1 else 0 end)  as ClaimsQueue,
        max(case when QueueName = 'INBOUND_Documents_Out' then 1 else 0 end) as DocumentsOutQueue,
        max(case when WrapupCodeName not in ('ININ-WRAP-UP-TIMEOUT', 'Default Wrap-Up Code')
                  and trim(WrapupCodeName) <> '' then WrapupCodeName end) as WrapupCodeName,
        sum(AcdWaitDurationMs)                                          as AcdWaitDurationMs,
        sum(Abandoned)                                                  as Abandoned
from    ${ads.ods}.CallSession
where   OriginatingDirection = 'inbound'
and     QueueName in ('INBOUND_HomeChangeOther', 'INBOUND_HomeChangeBuildCost', 'INBOUND_Claims', 'INBOUND_Documents_Out')
and     ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'
group by ConversationId

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_hinl_call_policy                                                            */
/*  Description : Each conversation with the caller's Home policy open at the call date (D-E1.24, share */
/*              : d step adsv2_call_open_policy; several open: the lowest code) and that policy's clien */
/*              : t.                                                                                    */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_hinl_call, stg.adsv2_call_open_policy, ods_v2.Policy                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_hinl_call_policy as
select  c.ConversationId, c.ConversationStartTime, c.CustomerPhoneKey, c.HomeChangeQueue, c.ClaimsQueue,
        c.DocumentsOutQueue, c.WrapupCodeName, c.AcdWaitDurationMs, c.Abandoned,
        p.ClientCode, o.PolicyCode, o.PolicyId
from    ${ads.stg}.adsv2_hinl_call c
left join ${ads.stg}.adsv2_call_open_policy o
        on  o.ConversationId = c.ConversationId and o.PolicyTypeGroup = 'Home'
left join ${ads.ods}.Policy p   on p.PolicyCode = o.PolicyCode

## Shared: hinl_arrears

Arrears chase sends (X6: Xtremepush events in MessageEvent, not ods.scvcampaignhistory): 'Arrears - Chase n' campaigns (mop-ups and TEST copies left out), sent by SMS or email, each matched to a Home policy: the policy on the message when it is a Home policy, else (no policy on the message) the client's Home policies open at the previous month-end. 04 used tmp.CurrentPolicy, every policy the client holds now (P8, P12).


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_hinl_arrears                                                                */
/*  Description : One row per arrears chase send and Home policy, with the chase number (1-6).          */
/*              :                                                                                       */
/*              : Reads: ods_v2.MessageEvent, ods_v2.Policy, ods_v2.PolicyMonthEnd                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_hinl_arrears as
select  distinct
        s.SentDateTime,
        s.ChaseNumber,
        s.ClientCode,
        coalesce(mp.PolicyCode, me.PolicyCode)                          as PolicyCode,
        coalesce(mp.PolicyId, me.PolicyId)                              as PolicyId
from    (
        select  EventDateTime                                           as SentDateTime,
                cast(regexp_extract(CampaignName, 'Chase ([0-9])', 1) as int) as ChaseNumber,
                ClientCode,
                PolicyCode                                              as MessagePolicyCode
        from    ${ads.ods}.MessageEvent
        where   CampaignName like 'Arrears - Chase %'
        and     lower(CampaignName) not like '%mop up%'
        and     InteractionType = 'sent'
        and     MessageType in ('EMAIL', 'SMS')
        and     EventDateTime >= add_months(timestamp '${ads.start_time}', -1) and EventDateTime < add_months(timestamp '${ads.end_time}', 1)
        ) s
left join (
        select  PolicyCode, PolicyId
        from    ${ads.ods}.Policy
        where   PolicyTypeGroup = 'Home'
        ) mp    on mp.PolicyCode = s.MessagePolicyCode
left join ${ads.ods}.PolicyMonthEnd me
        on  s.MessagePolicyCode is null
        and me.ClientCode = s.ClientCode
        and me.PolicyTypeGroup = 'Home'
        and me.OpenNum = 1
        and me.EffectiveDate = last_day(add_months(cast(s.SentDateTime as date), -1))
where   coalesce(mp.PolicyCode, me.PolicyCode) is not null

## HM1: Customer contacts Chill with a change request


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm1  (HM1)                                                              */
/*  Description : Mta / Spine. Notebook 04 cell 242. Inbound conversations on INBOUND_HomeChangeOther / */
/*              :  INBOUND_HomeChangeBuildCost, at their start. Change: callers on the pii Van phone li */
/*              : st are no longer dropped (R-1; a Home customer who also has a van was lost); source s */
/*              : ystem 20 Genesys (04 wrote 1).                                                        */
/*              :                                                                                       */
/*              : Grain: Call   Product: Home   Source: CallSession                                     */
/*              : Reads: stg.adsv2_hinl_call                                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm1 as
select  xxhash64('HM1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM1'                                               as EventTypeId,
        'Customer contacts Chill with a change request' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  ConversationStartTime                                           as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference
        from    ${ads.stg}.adsv2_hinl_call
        where   HomeChangeQueue = 1
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM1.F1: Call wait time-  or fails to make contact


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm1_f1  (HM1.F1)                                                        */
/*  Description : Mta / Friction. Notebook 04 cells 48, 243. Home change calls that waited more than 60 */
/*              : 0 seconds or were abandoned (D-E1.25; 04 took abandoned calls with any wait).         */
/*              :                                                                                       */
/*              : Grain: Call   Product: Home   Source: CallSession                                     */
/*              : Reads: stg.adsv2_hinl_call                                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm1_f1 as
select  xxhash64('HM1.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM1.F1'                                            as EventTypeId,
        'Call wait time-  or fails to make contact' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  ConversationStartTime                                           as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference
        from    ${ads.stg}.adsv2_hinl_call
        where   HomeChangeQueue = 1
        and     (AcdWaitDurationMs > 600000 or Abandoned > 0)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM2.O1: Cancellation-  declined or not proceeded with


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm2_o1  (HM2.O1)                                                        */
/*  Description : Mta / Outcome. Notebook 04 cell 244, corrected: a cancellation (R-5) within 30 days a */
/*              : fter a Home MTA posting (H19 default), at the cancel date, one per cancellation. 04 t */
/*              : ook any MTA policy whose status was ever 'Cancelled Mid Term' / 'Lapsed for Transfer' */
/*              : , at its latest cancel date, with no link to the change.                              */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, PolicyCancellation, Policy */
/*              : Reads: stg.adsv2_hinl_cancellation, stg.adsv2_hinl_mta                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm2_o1 as
select  xxhash64('HM2.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM2.O1'                                            as EventTypeId,
        'Cancellation-  declined or not proceeded with' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  c.CancelDateTime                                                as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference,
                max(c.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_cancellation c
        join    ${ads.stg}.adsv2_hinl_mta m     on m.PolicyCode = c.PolicyCode
        where   c.CancelDateTime >= m.PostingDateTime
        and     c.CancelDateTime <  m.PostingDateTime + interval 31 days
        and     c.CancelDateTime >= timestamp '${ads.start_time}' and c.CancelDateTime < timestamp '${ads.end_time}'
        group by c.PolicyCode, c.CancelDateTime
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM3: Customer accepts


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm3  (HM3)                                                              */
/*  Description : Mta / Spine. Notebook 04 cells 34, 245. The customer accepts the change: each Home MT */
/*              : A posting, at its posting date (X2; 04 kept the first posting per policy over the who */
/*              : le window). 04's status filter is not applied: it read every month-end snapshot, so i */
/*              : t never removed a row (E1, Motor MTA).                                                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, Policy                     */
/*              : Reads: stg.adsv2_hinl_mta                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm3 as
select  xxhash64('HM3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM3'                                               as EventTypeId,
        'Customer accepts' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  PostingDateTime                                                 as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_hinl_mta
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM4: Money due on the change


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm4  (HM4)                                                              */
/*  Description : Mta / Spine. Notebook 04 cell 247. Money due on the change: MTA postings that move pr */
/*              : emium or fees (either sign, as 04: return premiums included), one per posting.        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, Policy                     */
/*              : Reads: stg.adsv2_hinl_mta                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm4 as
select  xxhash64('HM4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM4'                                               as EventTypeId,
        'Money due on the change' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  PostingDateTime                                                 as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_hinl_mta
        where   GrossPremium <> 0 or Fees <> 0
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM4a: Paid now


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm4a  (HM4a)                                                            */
/*  Description : Mta / Spine. Notebook 04 cell 248. Paid now: the same rows as HM4, as 04 (no payment  */
/*              : route in the postings: PaymentType and PaymentMethod are 0 on every Home MTA posting  */
/*              : since 2025). Decision needed: drop, or a payment source.                              */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, Policy                     */
/*              : Reads: stg.adsv2_hinl_mta                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm4a as
select  xxhash64('HM4a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM4a'                                              as EventTypeId,
        'Paid now' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  PostingDateTime                                                 as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_hinl_mta
        where   GrossPremium <> 0 or Fees <> 0
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM4c: No money moves


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm4c  (HM4c)                                                            */
/*  Description : Mta / Spine. Notebook 04 cell 249. No money moves on the change (premium and fees bot */
/*              : h 0), one per posting.                                                                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, Policy                     */
/*              : Reads: stg.adsv2_hinl_mta                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm4c as
select  xxhash64('HM4c', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM4c'                                              as EventTypeId,
        'No money moves' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  PostingDateTime                                                 as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_hinl_mta
        where   GrossPremium = 0 and Fees = 0
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM6: Policy documents reissued


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm6  (HM6)                                                              */
/*  Description : Mta / Spine. Notebook 04 cell 250, anchored: the first document emailed or transmitte */
/*              : d on the policy within 7 days after an MTA posting (H18 default), one per posting. 04 */
/*              :  took the policy's last document of any kind in 5 years (P12, P15).                   */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, PolicyDocumentEvent        */
/*              : Reads: stg.adsv2_hinl_mta, stg.adsv2_relay_document_event                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm6 as
select  xxhash64('HM6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM6'                                               as EventTypeId,
        'Policy documents reissued' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_mta m
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = m.PolicyCode
        where   d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= m.PostingDateTime
        and     d.EventDateTime <  m.PostingDateTime + interval 8 days
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM6.B1.1: Policy documents dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm6_b1_1  (HM6.B1.1)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 258, anchored as HM6: 04's six new-business document n */
/*              : ames (terms of business, cover letter, suitability statement) within 7 days after the */
/*              :  posting, the last per posting. Decision needed (H18): these are new-business names;  */
/*              : Home MTA documents in Relay are 'Permanent MTA - ...' (about 1,000 a year).           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, PolicyDocumentEvent        */
/*              : Reads: stg.adsv2_hinl_mta, stg.adsv2_relay_document_event                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm6_b1_1 as
select  xxhash64('HM6.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM6.B1.1'                                          as EventTypeId,
        'Policy documents dispatched' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_mta m
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = m.PolicyCode
        where   d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= m.PostingDateTime
        and     d.EventDateTime <  m.PostingDateTime + interval 8 days
        and     d.EventDescription in ('New Business - Emailed Document - Terms Of Business',
                                       'New Business - Emailed Document - Cover Letter',
                                       'New Business - Emailed Document - 04 - Suitability Statement',
                                       'New Business - Document Transmitted - 04 - Suitability Statement',
                                       'New Business - Document Transmitted - Terms Of Business',
                                       'New Business - Document Transmitted - Cover Letter')
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM6.B1.2: Document pack dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm6_b1_2  (HM6.B1.2)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 259. As HM6.B1.1, the first per posting (when the pack */
/*              :  went out).                                                                           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, PolicyDocumentEvent        */
/*              : Reads: stg.adsv2_hinl_mta, stg.adsv2_relay_document_event                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm6_b1_2 as
select  xxhash64('HM6.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM6.B1.2'                                          as EventTypeId,
        'Document pack dispatched' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_mta m
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = m.PolicyCode
        where   d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= m.PostingDateTime
        and     d.EventDateTime <  m.PostingDateTime + interval 8 days
        and     d.EventDescription in ('New Business - Emailed Document - Terms Of Business',
                                       'New Business - Emailed Document - Cover Letter',
                                       'New Business - Emailed Document - 04 - Suitability Statement',
                                       'New Business - Document Transmitted - 04 - Suitability Statement',
                                       'New Business - Document Transmitted - Terms Of Business',
                                       'New Business - Document Transmitted - Cover Letter')
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM4.B1.F1: Chases before the customer submits-  and documents never submitted


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm4_b1_f1  (HM4.B1.F1)                                                  */
/*  Description : Mta / Friction. Notebook 04 cell 246, anchored: one event per chase sent with one of  */
/*              : seven documents outstanding within 30 days after an MTA posting, at its send date. 04 */
/*              :  took every chase of an MTA policy in the window at the sale date, one per policy.    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, DocumentChaseHome          */
/*              : Reads: stg.adsv2_hinl_chase, stg.adsv2_hinl_mta                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm4_b1_f1 as
select  xxhash64('HM4.B1.F1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM4.B1.F1'                                         as EventTypeId,
        'Chases before the customer submits-  and documents never submitted' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  c.SendDate                                                      as EventDateTime,
                1                                                               as SourceSystemId,
                c.Campaign                                                      as EventSourceReference,
                c.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_chase c
        join    ${ads.stg}.adsv2_hinl_mta m     on m.PolicyCode = c.PolicyCode
        where   c.SevenOutstanding = 1
        and     c.SendDate >= m.PostingDateTime
        and     c.SendDate <  m.PostingDateTime + interval 31 days
        and     c.SendDate >= timestamp '${ads.start_time}' and c.SendDate < timestamp '${ads.end_time}'
        group by c.PolicyCode, c.Campaign, c.SendDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM4.B1.1: Document request issued


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm4_b1_1  (HM4.B1.1)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 251, corrected (D-E1.16): the DAY 1 chase send (reques */
/*              : t issued) with one of six documents outstanding, new-business snapshot rows as 04, wi */
/*              : thin 30 days after an MTA posting, one per posting.                                   */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, DocumentChaseHome          */
/*              : Reads: stg.adsv2_hinl_chase, stg.adsv2_hinl_mta                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm4_b1_1 as
select  xxhash64('HM4.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM4.B1.1'                                          as EventTypeId,
        'Document request issued' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(c.SendDate)                                                 as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_chase c
        join    ${ads.stg}.adsv2_hinl_mta m     on m.PolicyCode = c.PolicyCode
        where   c.PolicyType = 'New Business'
        and     c.Campaign = 'DAY 1'
        and     c.SixOutstanding = 1
        and     c.SendDate >= m.PostingDateTime
        and     c.SendDate <  m.PostingDateTime + interval 31 days
        and     c.SendDate >= timestamp '${ads.start_time}' and c.SendDate < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM4.B1.4: Chase-  first reminder


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm4_b1_4  (HM4.B1.4)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 253, corrected (D-E1.16): the first reminder is the DA */
/*              : Y 7 chase send (04 read DAY 1, the same rows as HM4.B1.1) with one of six documents o */
/*              : utstanding, within 30 days after an MTA posting, one per posting.                     */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, DocumentChaseHome          */
/*              : Reads: stg.adsv2_hinl_chase, stg.adsv2_hinl_mta                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm4_b1_4 as
select  xxhash64('HM4.B1.4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM4.B1.4'                                          as EventTypeId,
        'Chase-  first reminder' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(c.SendDate)                                                 as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_chase c
        join    ${ads.stg}.adsv2_hinl_mta m     on m.PolicyCode = c.PolicyCode
        where   c.Campaign = 'DAY 7'
        and     c.SixOutstanding = 1
        and     c.SendDate >= m.PostingDateTime
        and     c.SendDate <  m.PostingDateTime + interval 31 days
        and     c.SendDate >= timestamp '${ads.start_time}' and c.SendDate < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM4.B1.5: Chase-  escalation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm4_b1_5  (HM4.B1.5)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 254, corrected (D-E1.16): the escalation is the Relay  */
/*              : final chase (the first 'Chase Final Notice' email or letter sent) within 30 days afte */
/*              : r an MTA posting, one per posting. 04 read the snapshot's DAY 20 rows at the sale dat */
/*              : e.                                                                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, PolicyDocumentEvent        */
/*              : Reads: stg.adsv2_hinl_mta, stg.adsv2_relay_document_event                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm4_b1_5 as
select  xxhash64('HM4.B1.5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM4.B1.5'                                          as EventTypeId,
        'Chase-  escalation' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_mta m
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = m.PolicyCode
        where   d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= m.PostingDateTime
        and     d.EventDateTime <  m.PostingDateTime + interval 31 days
        and     d.EventDescription like '%Chase Final Notice%'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%'
                 or d.EventDescription like '%Document Posted Confirmation%')
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM4.B1.2: Customer submits documents


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm4_b1_2  (HM4.B1.2)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 252, anchored. The first document upload within 30 day */
/*              : s after an MTA posting, one per posting (04 took any upload in the window).           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, DocumentUpload             */
/*              : Reads: stg.adsv2_hinl_mta, ods_v2.DocumentUpload                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm4_b1_2 as
select  xxhash64('HM4.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM4.B1.2'                                          as EventTypeId,
        'Customer submits documents' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(u.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_mta m
        join    ${ads.ods}.DocumentUpload u     on u.PolicyCode = m.PolicyCode
        where   cast(u.EventTimestamp as timestamp) >= m.PostingDateTime
        and     cast(u.EventTimestamp as timestamp) <  m.PostingDateTime + interval 31 days
        and     cast(u.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(u.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM4.B1.3: Documents validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm4_b1_3  (HM4.B1.3)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 255, anchored. The last MyChill document-status record */
/*              :  (accepted or not, as 04) within 30 days after an MTA posting, one per posting.       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, DocumentWorkflowStatus     */
/*              : Reads: stg.adsv2_hinl_mta, ods_v2.DocumentWorkflowStatus                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm4_b1_3 as
select  xxhash64('HM4.B1.3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM4.B1.3'                                          as EventTypeId,
        'Documents validated' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(cast(w.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_mta m
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = m.PolicyCode
        where   cast(w.EventTimestamp as timestamp) >= m.PostingDateTime
        and     cast(w.EventTimestamp as timestamp) <  m.PostingDateTime + interval 31 days
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM4.B1.6: Documents received and validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm4_b1_6  (HM4.B1.6)                                                    */
/*  Description : Mta / Branch. Notebook 04 cell 257, anchored. The last accepted document within 30 da */
/*              : ys after an MTA posting, one per posting.                                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, DocumentWorkflowStatus     */
/*              : Reads: stg.adsv2_hinl_mta, ods_v2.DocumentWorkflowStatus                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm4_b1_6 as
select  xxhash64('HM4.B1.6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM4.B1.6'                                          as EventTypeId,
        'Documents received and validated' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(cast(w.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_mta m
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = m.PolicyCode
        where   w.IsAccepted = true
        and     cast(w.EventTimestamp as timestamp) >= m.PostingDateTime
        and     cast(w.EventTimestamp as timestamp) <  m.PostingDateTime + interval 31 days
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HM4.B1.O1: Cancellation-  non-receipt


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hm4_b1_o1  (HM4.B1.O1)                                                  */
/*  Description : Mta / Outcome. Notebook 04 cell 256, anchored: a ' CXL ' document emailed or transmit */
/*              : ted within 60 days after an MTA posting that had a chase with one of six documents ou */
/*              : tstanding in its 30 days; the last per posting. 04 tied the CXL document, the chase a */
/*              : nd the MTA only by policy code.                                                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyTransaction, DocumentChaseHome, PolicyD */
/*              : ocumentEvent                                                                          */
/*              : Reads: stg.adsv2_hinl_mta, stg.adsv2_relay_document_event, stg.adsv2_hinl_chase       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hm4_b1_o1 as
select  xxhash64('HM4.B1.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HM4.B1.O1'                                         as EventTypeId,
        'Cancellation-  non-receipt' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                m.PolicyCode                                                    as SourcePolicyReference,
                max(m.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_mta m
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = m.PolicyCode
        left semi join ${ads.stg}.adsv2_hinl_chase c
                on  c.PolicyCode = m.PolicyCode
                and c.SixOutstanding = 1
                and c.SendDate >= m.PostingDateTime
                and c.SendDate <  m.PostingDateTime + interval 31 days
        where   d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= m.PostingDateTime
        and     d.EventDateTime <  m.PostingDateTime + interval 61 days
        and     d.EventDescription like '% CXL %'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by m.PolicyCode, m.PostingDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HC1a: Customer asks to cancel


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hc1a  (HC1a)                                                            */
/*  Description : Cancellation / Spine. Notebook 04 cell 260, corrected (R-5): Home cancellations with  */
/*              : a customer / client reason, at the cancel date (when it took effect, not when the cus */
/*              : tomer asked), one per cancellation. 04 joined the mtc table on PolicyCode (the CXL re */
/*              : cord, not the cancelled policy) and kept only rows with PolicyCancelFlag 'Y' in stg.J */
/*              : ulyPolicyState.                                                                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyCancellation                            */
/*              : Reads: stg.adsv2_hinl_cancellation                                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hc1a as
select  xxhash64('HC1a', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HC1a'                                              as EventTypeId,
        'Customer asks to cancel' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CancelDateTime                                                  as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_hinl_cancellation
        where   (Reason like 'Client%' or Reason like 'Customer%')
        and     CancelDateTime >= timestamp '${ads.start_time}' and CancelDateTime < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HC1a.F1: No retention or save attempt anywhere


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hc1a_f1  (HC1a.F1)                                                      */
/*  Description : Cancellation / Friction. Notebook 04 cell 261, as 04: cancellations whose reason ment */
/*              : ions NCT, at the cancel date. The step is "no retention or save attempt", and Home ha */
/*              : s no NCT, so expect no rows (decision needed: drop or a real source).                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyCancellation                            */
/*              : Reads: stg.adsv2_hinl_cancellation                                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hc1a_f1 as
select  xxhash64('HC1a.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HC1a.F1'                                           as EventTypeId,
        'No retention or save attempt anywhere' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  CancelDateTime                                                  as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_hinl_cancellation
        where   Reason like '%NCT%'
        and     CancelDateTime >= timestamp '${ads.start_time}' and CancelDateTime < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HC2: Document chase-  reminder


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hc2  (HC2)                                                              */
/*  Description : Cancellation / Spine. Notebook 04 cell 262, anchored (R-5): the first chase sent with */
/*              :  one of six documents outstanding in the 60 days before the cancellation, at its send */
/*              :  date, one per cancellation. 04 joined the chase snapshot by client code to any cance */
/*              : lled client, at the sale date.                                                        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyCancellation, DocumentChaseHome         */
/*              : Reads: stg.adsv2_hinl_cancellation, stg.adsv2_hinl_chase                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hc2 as
select  xxhash64('HC2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HC2'                                               as EventTypeId,
        'Document chase-  reminder' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(h.SendDate)                                                 as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference,
                max(c.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_cancellation c
        join    ${ads.stg}.adsv2_hinl_chase h   on h.PolicyCode = c.PolicyCode
        where   h.SixOutstanding = 1
        and     h.SendDate >= c.CancelDateTime - interval 60 days
        and     h.SendDate <= c.CancelDateTime
        and     h.SendDate >= timestamp '${ads.start_time}' and h.SendDate < timestamp '${ads.end_time}'
        group by c.PolicyCode, c.CancelDateTime
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HC3: Document chase-  escalation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hc3  (HC3)                                                              */
/*  Description : Cancellation / Spine. Notebook 04 cell 263, anchored as HC2. The first escalation cha */
/*              : se (any campaign after DAY 1) in the 60 days before the cancellation.                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyCancellation, DocumentChaseHome         */
/*              : Reads: stg.adsv2_hinl_cancellation, stg.adsv2_hinl_chase                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hc3 as
select  xxhash64('HC3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HC3'                                               as EventTypeId,
        'Document chase-  escalation' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(h.SendDate)                                                 as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference,
                max(c.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_cancellation c
        join    ${ads.stg}.adsv2_hinl_chase h   on h.PolicyCode = c.PolicyCode
        where   h.SixOutstanding = 1
        and     h.Campaign <> 'DAY 1'
        and     h.SendDate >= c.CancelDateTime - interval 60 days
        and     h.SendDate <= c.CancelDateTime
        and     h.SendDate >= timestamp '${ads.start_time}' and h.SendDate < timestamp '${ads.end_time}'
        group by c.PolicyCode, c.CancelDateTime
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HC4: Final notice


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hc4  (HC4)                                                              */
/*  Description : Cancellation / Spine. Notebook 04 cell 264, corrected (H21 default): the first 'Docum */
/*              : ent Chase ... Final Notice' emailed or transmitted on the policy in the 30 days befor */
/*              : e the cancellation, one per cancellation. 04 took the last final notice of any policy */
/*              :  of a cancelled client, at any time.                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyCancellation, PolicyDocumentEvent       */
/*              : Reads: stg.adsv2_hinl_cancellation, stg.adsv2_relay_document_event                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hc4 as
select  xxhash64('HC4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HC4'                                               as EventTypeId,
        'Final notice' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference,
                max(c.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_cancellation c
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = c.PolicyCode
        where   d.PolicyTypeGroup = 'Home'
        and     d.EventDescription like '%Document Chase%'
        and     d.EventDescription like '%Final Notice%'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDateTime >= c.CancelDateTime - interval 30 days
        and     d.EventDateTime <= c.CancelDateTime + interval 1 days
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by c.PolicyCode, c.CancelDateTime
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HC5: Cancellation processed


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hc5  (HC5)                                                              */
/*  Description : Cancellation / Spine. Notebook 04 cell 265, anchored (R-5): the last 'CXL' document e */
/*              : mailed or transmitted on the cancelled policy or its CXL record within 30 days either */
/*              :  side of the cancellation, one per cancellation. 04 took the last CXL document of any */
/*              :  policy of a cancelled client.                                                        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyCancellation, PolicyDocumentEvent       */
/*              : Reads: stg.adsv2_hinl_cancellation, stg.adsv2_relay_document_event                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hc5 as
select  xxhash64('HC5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HC5'                                               as EventTypeId,
        'Cancellation processed' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference,
                max(c.PolicyId)                                                 as PolicyId
        from    (
                select  PolicyCode, PolicyId, CancelDateTime, PolicyCode as DocPolicyCode from ${ads.stg}.adsv2_hinl_cancellation
                union all
                select  PolicyCode, PolicyId, CancelDateTime, CxlPolicyCode from ${ads.stg}.adsv2_hinl_cancellation where CxlPolicyCode is not null
                ) c
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = c.DocPolicyCode
        where   d.PolicyTypeGroup = 'Home'
        and     d.EventDescription like '%CXL%'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%')
        and     d.EventDateTime >= c.CancelDateTime - interval 30 days
        and     d.EventDateTime <= c.CancelDateTime + interval 30 days
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by c.PolicyCode, c.CancelDateTime
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HC6: Cancellation confirmation sent


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hc6  (HC6)                                                              */
/*  Description : Cancellation / Spine. Notebook 04 cell 266 (mirrored from VC6), anchored (R-5): the f */
/*              : irst insurer-led cancellation email ('Insurer Led Cancelation - Emailed Document - Re */
/*              : g canx email template', Relay's spelling) within 30 days either side of a cancellatio */
/*              : n of that policy. 04 took every such email, cancelled or not.                         */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyCancellation, PolicyDocumentEvent       */
/*              : Reads: stg.adsv2_hinl_cancellation, stg.adsv2_relay_document_event                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hc6 as
select  xxhash64('HC6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HC6'                                               as EventTypeId,
        'Cancellation confirmation sent' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                c.PolicyCode                                                    as SourcePolicyReference,
                max(c.PolicyId)                                                 as PolicyId
        from    (
                select  PolicyCode, PolicyId, CancelDateTime, PolicyCode as DocPolicyCode from ${ads.stg}.adsv2_hinl_cancellation
                union all
                select  PolicyCode, PolicyId, CancelDateTime, CxlPolicyCode from ${ads.stg}.adsv2_hinl_cancellation where CxlPolicyCode is not null
                ) c
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = c.DocPolicyCode
        where   d.PolicyTypeGroup = 'Home'
        and     d.EventDescription = 'Insurer Led Cancelation - Emailed Document - Reg canx email template'
        and     d.EventDateTime >= c.CancelDateTime - interval 30 days
        and     d.EventDateTime <= c.CancelDateTime + interval 30 days
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by c.PolicyCode, c.CancelDateTime
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HARR.1: Missed payment detected


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_harr_1  (HARR.1)                                                        */
/*  Description : Arrears / Spine. Notebook 04 cell 267. Missed payment detected: each 'Arrears - Chase */
/*              :  1 - 1 Day Past' send, one event per send and policy (X2; 04 kept the latest send per */
/*              :  policy over the window).                                                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: MessageEvent, PolicyMonthEnd, Policy          */
/*              : Reads: stg.adsv2_hinl_arrears                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_harr_1 as
select  xxhash64('HARR.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HARR.1'                                            as EventTypeId,
        'Missed payment detected' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  SentDateTime                                                    as EventDateTime,
                21                                                              as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                max(PolicyId)                                                   as PolicyId
        from    ${ads.stg}.adsv2_hinl_arrears
        where   ChaseNumber = 1
        and     SentDateTime >= timestamp '${ads.start_time}' and SentDateTime < timestamp '${ads.end_time}'
        group by PolicyCode, SentDateTime
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HARR.2: Arrears chase


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_harr_2  (HARR.2)                                                        */
/*  Description : Arrears / Spine. Notebook 04 cell 268, on the Motor rule (D-E1.9, H22 default): every */
/*              :  arrears chase send 1-5, one event per send and policy. 04 read Chase 1 only (the sam */
/*              : e rows as HARR.1).                                                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: MessageEvent, PolicyMonthEnd, Policy          */
/*              : Reads: stg.adsv2_hinl_arrears                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_harr_2 as
select  xxhash64('HARR.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HARR.2'                                            as EventTypeId,
        'Arrears chase' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  SentDateTime                                                    as EventDateTime,
                21                                                              as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                max(PolicyId)                                                   as PolicyId
        from    ${ads.stg}.adsv2_hinl_arrears
        where   ChaseNumber between 1 and 5
        and     SentDateTime >= timestamp '${ads.start_time}' and SentDateTime < timestamp '${ads.end_time}'
        group by PolicyCode, SentDateTime
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HARR.O1: Arrears cleared


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_harr_o1  (HARR.O1)                                                      */
/*  Description : Arrears / Outcome. Notebook 04 cell 269, made reproducible: arrears cleared = a Chase */
/*              :  1 send with no Chase 5 for the policy in the 35 days after it, once those 35 days ar */
/*              : e inside the window; at the Chase 1 send. 04 compared with the run date (current_time */
/*              : stamp - 28 days) and with any Chase 5 anywhere in the window.                         */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: MessageEvent, PolicyMonthEnd, Policy          */
/*              : Reads: stg.adsv2_hinl_arrears                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_harr_o1 as
select  xxhash64('HARR.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HARR.O1'                                           as EventTypeId,
        'Arrears cleared' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  a.SentDateTime                                                  as EventDateTime,
                21                                                              as SourceSystemId,
                a.PolicyCode                                                    as SourcePolicyReference,
                max(a.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_hinl_arrears a
        left anti join (
                select  PolicyCode, SentDateTime
                from    ${ads.stg}.adsv2_hinl_arrears
                where   ChaseNumber = 5
                ) f     on  f.PolicyCode = a.PolicyCode
                        and f.SentDateTime > a.SentDateTime
                        and f.SentDateTime <= a.SentDateTime + interval 35 days
        where   a.ChaseNumber = 1
        and     a.SentDateTime + interval 35 days < timestamp '${ads.end_time}'
        and     a.SentDateTime >= timestamp '${ads.start_time}' and a.SentDateTime < timestamp '${ads.end_time}'
        group by a.PolicyCode, a.SentDateTime
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HARR.3: Loan defaults-  arrears not cleared


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_harr_3  (HARR.3)                                                        */
/*  Description : Arrears / Spine. Notebook 04 cell 270. Arrears not cleared: each 'Arrears - Chase 5 - */
/*              :  28 Days Past' send (a chase, not the loan default: no loan data in ods_v2), one even */
/*              : t per send and policy.                                                                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: MessageEvent, PolicyMonthEnd, Policy          */
/*              : Reads: stg.adsv2_hinl_arrears                                                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_harr_3 as
select  xxhash64('HARR.3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HARR.3'                                            as EventTypeId,
        'Loan defaults-  arrears not cleared' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  SentDateTime                                                    as EventDateTime,
                21                                                              as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                max(PolicyId)                                                   as PolicyId
        from    ${ads.stg}.adsv2_hinl_arrears
        where   ChaseNumber = 5
        and     SentDateTime >= timestamp '${ads.start_time}' and SentDateTime < timestamp '${ads.end_time}'
        group by PolicyCode, SentDateTime
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HARR.O2: Escalation to forced cancellation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_harr_o2  (HARR.O2)                                                      */
/*  Description : Arrears / Outcome. Notebook 04 cell 271, anchored: the first Relay 'Insurer Led ...'  */
/*              : document on the policy within 60 days after an arrears chase send, one per policy and */
/*              :  document date. 04 took any insurer-led event on a policy that ever had an arrears ca */
/*              : mpaign.                                                                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: MessageEvent, PolicyMonthEnd, PolicyDocumentE */
/*              : vent                                                                                  */
/*              : Reads: stg.adsv2_relay_document_event, stg.adsv2_hinl_arrears                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_harr_o2 as
select  xxhash64('HARR.O2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HARR.O2'                                           as EventTypeId,
        'Escalation to forced cancellation' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                d.PolicyCode                                                    as SourcePolicyReference,
                max(d.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_relay_document_event d
        left semi join ${ads.stg}.adsv2_hinl_arrears a
                on  a.PolicyCode = d.PolicyCode
                and d.EventDateTime >= a.SentDateTime
                and d.EventDateTime <  a.SentDateTime + interval 61 days
        where   d.PolicyTypeGroup = 'Home'
        and     d.EventDescription like 'Insurer Led%'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        group by d.PolicyCode, cast(d.EventDateTime as date)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HCL1a: Home claim reported


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hcl1a  (HCL1a)                                                          */
/*  Description : Claims / Spine. Notebook 04 cells 62, 64, 272. Inbound INBOUND_Claims conversations w */
/*              : hose caller (phone key -> Relay client) held an open Home policy at the previous mont */
/*              : h-end, one per conversation, at its start. 04 matched raw phones through a pii list a */
/*              : nd kept the first call per policy over the window. Source system 20 Genesys.          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: CallSession, SourceCustomer, PolicyMonthEnd   */
/*              : Reads: stg.adsv2_hinl_call_policy                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hcl1a as
select  xxhash64('HCL1a', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HCL1a'                                             as EventTypeId,
        'Home claim reported' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  ConversationStartTime                                           as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                ClientCode                                                      as SourceCustomerReference,
                'ClientCode'                                                    as SourceCustomerReferenceType,
                PolicyId
        from    ${ads.stg}.adsv2_hinl_call_policy
        where   ClaimsQueue = 1
        and     PolicyCode is not null
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HCL1b: Escape of water or emergency assistance


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hcl1b  (HCL1b)                                                          */
/*  Description : Claims / Spine. Notebook 04 cell 273. Home QS claims of type 'Escape Of Water': one e */
/*              : vent per claim, at the first Home QS transaction on its risk in the window (04: the f */
/*              : irst per policy). Source system 4 Home QS. Decision needed (H23): these claim details */
/*              :  look like declared history, not claims during cover.                                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyClaimHome, PolicyHomeQsEvent            */
/*              : Reads: ods_v2.PolicyClaimHome, ods_v2.PolicyHomeQsEvent                               */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hcl1b as
select  xxhash64('HCL1b', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HCL1b'                                             as EventTypeId,
        'Escape of water or emergency assistance' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(e.TransactionDate as timestamp))                       as EventDateTime,
                4                                                               as SourceSystemId,
                c.HomeClaimId                                                   as EventSourceReference,
                e.PolicyCode                                                    as SourcePolicyReference,
                max(e.PolicyId)                                                 as PolicyId
        from    ${ads.ods}.PolicyClaimHome c
        join    ${ads.ods}.PolicyHomeQsEvent e  on e.HomeRiskId = c.HomeRiskId
        where   c.HomeClaimType = 'Escape Of Water'
        and     cast(e.TransactionDate as timestamp) >= timestamp '${ads.start_time}' and cast(e.TransactionDate as timestamp) < timestamp '${ads.end_time}'
        group by e.PolicyCode, c.HomeClaimId
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HCL1c: Storm-  flood or weather claim


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hcl1c  (HCL1c)                                                          */
/*  Description : Claims / Spine. Notebook 04 cell 274. As HCL1b for 'Storm Damage' claims.             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyClaimHome, PolicyHomeQsEvent            */
/*              : Reads: ods_v2.PolicyClaimHome, ods_v2.PolicyHomeQsEvent                               */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hcl1c as
select  xxhash64('HCL1c', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HCL1c'                                             as EventTypeId,
        'Storm-  flood or weather claim' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'derived'                                           as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(e.TransactionDate as timestamp))                       as EventDateTime,
                4                                                               as SourceSystemId,
                c.HomeClaimId                                                   as EventSourceReference,
                e.PolicyCode                                                    as SourcePolicyReference,
                max(e.PolicyId)                                                 as PolicyId
        from    ${ads.ods}.PolicyClaimHome c
        join    ${ads.ods}.PolicyHomeQsEvent e  on e.HomeRiskId = c.HomeRiskId
        where   c.HomeClaimType = 'Storm Damage'
        and     cast(e.TransactionDate as timestamp) >= timestamp '${ads.start_time}' and cast(e.TransactionDate as timestamp) < timestamp '${ads.end_time}'
        group by e.PolicyCode, c.HomeClaimId
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HCL.O2: Policy ended


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hcl_o2  (HCL.O2)                                                        */
/*  Description : Claims / Outcome. Notebook 04 cell 275. A Home QS 'Policy Cancelled' transaction on a */
/*              :  claim risk of a renewing home policy (the Home renewal base), one per policy and tra */
/*              : nsaction date (04: the first per policy, window end 2026-08-01 hard-coded). Source sy */
/*              : stem 4 Home QS.                                                                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicyClaimHome, PolicyHomeQsEvent, PolicyRen */
/*              : ewal                                                                                  */
/*              : Reads: ods_v2.PolicyHomeQsEvent, ods_v2.PolicyClaimHome, stg.adsv2_renewal_base       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hcl_o2 as
select  xxhash64('HCL.O2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HCL.O2'                                            as EventTypeId,
        'Policy ended' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(e.TransactionDate as timestamp)                            as EventDateTime,
                4                                                               as SourceSystemId,
                e.PolicyCode                                                    as SourcePolicyReference,
                max(e.PolicyId)                                                 as PolicyId
        from    ${ads.ods}.PolicyHomeQsEvent e
        left semi join ${ads.ods}.PolicyClaimHome c on c.HomeRiskId = e.HomeRiskId
        join    (select distinct PolicyCode from ${ads.stg}.adsv2_renewal_base where LYPolicyTypeGroup = 'Home') b
                on b.PolicyCode = e.PolicyCode
        where   e.EventType = 'Policy Cancelled'
        and     cast(e.TransactionDate as timestamp) >= add_months(timestamp '${ads.start_time}', -1) and cast(e.TransactionDate as timestamp) < timestamp '${ads.end_time}'
        group by e.PolicyCode, e.TransactionDate
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HD1: Customer rings and asks for a document


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hd1  (HD1)                                                              */
/*  Description : Doc Requests / Spine. Notebook 04 cell 276, corrected (D-E1.5, H24 default): each inb */
/*              : ound INBOUND_Documents_Out conversation that is a Home request (a Home-named wrap-up  */
/*              : code such as 'LOI Home Request', or a caller holding an open Home policy and no Motor */
/*              : -named code), transfers left out, at the call start, with the caller's Home policy. 0 */
/*              : 4 kept only calls followed by a Relay document within 2 days, Jul-Aug 2026 only, one  */
/*              : per policy.                                                                           */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: CallSession, SourceCustomer, PolicyMonthEnd   */
/*              : Reads: stg.adsv2_hinl_call_policy                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hd1 as
select  xxhash64('HD1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HD1'                                               as EventTypeId,
        'Customer rings and asks for a document' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  ConversationStartTime                                           as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                ClientCode                                                      as SourceCustomerReference,
                case when ClientCode is not null then 'ClientCode' end          as SourceCustomerReferenceType,
                PolicyId
        from    ${ads.stg}.adsv2_hinl_call_policy
        where   DocumentsOutQueue = 1
        and     coalesce(WrapupCodeName, '') not like '%Transfer%'
        and     (WrapupCodeName like '%Home%'
                 or (PolicyCode is not null and coalesce(WrapupCodeName, '') not like '%Motor%'))
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HD1.F1: Call wait time-  or fails to make contact


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hd1_f1  (HD1.F1)                                                        */
/*  Description : Doc Requests / Friction. Notebook 04 cell 277, corrected (D-E1.25): Home document-req */
/*              : uest calls (as HD1) that waited more than 600 seconds or were abandoned. 04 took aban */
/*              : doned calls of clients with a Relay document in Jul-Aug 2026, at any date.            */
/*              :                                                                                       */
/*              : Grain: Call   Product: Home   Source: CallSession, SourceCustomer, PolicyMonthEnd     */
/*              : Reads: stg.adsv2_hinl_call_policy                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hd1_f1 as
select  xxhash64('HD1.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HD1.F1'                                            as EventTypeId,
        'Call wait time-  or fails to make contact' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  ConversationStartTime                                           as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                PolicyId
        from    ${ads.stg}.adsv2_hinl_call_policy
        where   DocumentsOutQueue = 1
        and     coalesce(WrapupCodeName, '') not like '%Transfer%'
        and     (WrapupCodeName like '%Home%'
                 or (PolicyCode is not null and coalesce(WrapupCodeName, '') not like '%Motor%'))
        and     (AcdWaitDurationMs > 600000 or Abandoned > 0)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HD3.F1: Non-arrival-  and the customer rings again


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_hd3_f1  (HD3.F1)                                                        */
/*  Description : Doc Requests / Friction. Notebook 04 cell 279, corrected (D-E1.8): the customer rings */
/*              :  again, a Home document-request call by the same client within 14 days of their previ */
/*              : ous one, at the repeat call. 04 took clients with calls on more than one date in Jul- */
/*              : Aug 2026, at their last call.                                                         */
/*              :                                                                                       */
/*              : Grain: Customer   Product: Home   Source: CallSession, SourceCustomer, PolicyMonthEnd */
/*              : Reads: stg.adsv2_hinl_call_policy                                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_hd3_f1 as
select  xxhash64('HD3.F1', 'Customer',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HD3.F1'                                            as EventTypeId,
        'Non-arrival-  and the customer rings again' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Customer'                                          as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  ConversationStartTime                                           as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                ClientCode                                                      as SourceCustomerReference,
                'ClientCode'                                                    as SourceCustomerReferenceType,
                PolicyId
        from    (
                select  *, lag(ConversationStartTime) over (partition by ClientCode order by ConversationStartTime, ConversationId) as PreviousCall
                from    ${ads.stg}.adsv2_hinl_call_policy
                where   DocumentsOutQueue = 1
                and     ClientCode is not null
                and     coalesce(WrapupCodeName, '') not like '%Transfer%'
                and     (WrapupCodeName like '%Home%'
                         or (PolicyCode is not null and coalesce(WrapupCodeName, '') not like '%Motor%'))
                ) x
        where   PreviousCall is not null
        and     ConversationStartTime <= PreviousCall + interval 14 days
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

In [ ]:
%%pyspark
# Events per family (a family with no events is reported, not failed: check its source and window)
for t in ["adsv2_evt_hm1", "adsv2_evt_hm1_f1", "adsv2_evt_hm2_o1", "adsv2_evt_hm3", "adsv2_evt_hm4", "adsv2_evt_hm4a", "adsv2_evt_hm4c", "adsv2_evt_hm6", "adsv2_evt_hm6_b1_1", "adsv2_evt_hm6_b1_2", "adsv2_evt_hm4_b1_f1", "adsv2_evt_hm4_b1_1", "adsv2_evt_hm4_b1_4", "adsv2_evt_hm4_b1_5", "adsv2_evt_hm4_b1_2", "adsv2_evt_hm4_b1_3", "adsv2_evt_hm4_b1_6", "adsv2_evt_hm4_b1_o1", "adsv2_evt_hc1a", "adsv2_evt_hc1a_f1", "adsv2_evt_hc2", "adsv2_evt_hc3", "adsv2_evt_hc4", "adsv2_evt_hc5", "adsv2_evt_hc6", "adsv2_evt_harr_1", "adsv2_evt_harr_2", "adsv2_evt_harr_o1", "adsv2_evt_harr_3", "adsv2_evt_harr_o2", "adsv2_evt_hcl1a", "adsv2_evt_hcl1b", "adsv2_evt_hcl1c", "adsv2_evt_hcl_o2", "adsv2_evt_hd1", "adsv2_evt_hd1_f1", "adsv2_evt_hd3_f1"]:
    n = spark.table(f"{stg_schema}.{t}").count()
    print(f"{t:<40} {n:>12,}" + ("   <- no events" if n == 0 else ""))
ads_run_log("16_Home_InLife", "succeeded")
